# PCB Defect Detection — YOLOv8 Training on Google Colab (Free T4 GPU)

This notebook trains the **Phase 1 Source-Domain YOLOv8n model** on the complete **PKU-Market-PCB dataset** (~3,326 images, 6 defect classes).

### Estimated Training Time:
- **Local CPU:** ~8 to 9 hours
- **Google Colab Free T4 GPU:** **~10 to 15 minutes** (50 epochs)

---

### Step 1: Verify GPU Acceleration
Ensure your Colab runtime is set to GPU:
1. Go to menu: **Runtime** -> **Change runtime type**
2. Select **T4 GPU** under Hardware accelerator -> Click **Save**

In [ ]:
!nvidia-smi

### Step 2: Install Required Libraries

In [ ]:
!pip install -q ultralytics roboflow

### Step 3: Download Dataset from Roboflow Universe

In [ ]:
import os
from roboflow import Roboflow

# Pre-configured with your Roboflow API key
rf = Roboflow(api_key="Txn9rCczL5tK1n0PYkpx")
project = rf.workspace("jr-mqqnk").project("pcb-defects-detection-anddl")
dataset = project.version(1).download("yolov8")

print("\n[OK] Dataset downloaded successfully to:", dataset.location)
data_yaml_path = os.path.join(dataset.location, "data.yaml")

### Step 4: Train YOLOv8n on GPU (50 Epochs)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Train with GPU acceleration
results = model.train(
    data=data_yaml_path,
    epochs=50,
    batch=16,
    imgsz=640,
    patience=10,          # Early stopping if no improvement for 10 epochs
    device=0,             # GPU 0
    workers=4,
    seed=42,
    project="runs",
    name="source_model_colab",
    exist_ok=True,
    pretrained=True,
    fliplr=0.5,
    degrees=10.0,
    scale=0.3,
    mosaic=1.0,
    close_mosaic=10,
)

print("\n[OK] Training completed on GPU!")

### Step 5: Evaluate Model on Held-Out Test Split

In [ ]:
# Run evaluation strictly on the held-out test split
test_metrics = model.val(data=data_yaml_path, split="test", imgsz=640, device=0)

print("\n" + "=" * 50)
print(f"Test Set mAP@0.5:      {test_metrics.box.map50:.4f}")
print(f"Test Set mAP@0.5:0.95: {test_metrics.box.map:.4f}")
print(f"Mean Precision:       {test_metrics.box.mp:.4f}")
print(f"Mean Recall:          {test_metrics.box.mr:.4f}")
print("=" * 50)

### Step 6: Download Trained Weights & Results to Your PC
This packages your `best.pt`, `results.csv`, confusion matrix, and training curves into a zip and triggers a browser download.

In [ ]:
import shutil
from google.colab import files

# Create zip archive of the training outputs
shutil.make_archive("pcb_source_model_weights", "zip", "runs/source_model_colab")

# Trigger browser download
files.download("pcb_source_model_weights.zip")
print("Download initiated! Extract the zip into your local project.")